In [ ]:
import sys
sys.path.append('../')

In [ ]:

from datasets import load_dataset
def load_xsum():
    print("Loading XSum dataset...")
    # Load the dataset from the specified path
    dataset = load_dataset("EdinburghNLP/xsum", split="train")
    return dataset

# load_xsum()

In [ ]:

def load_owt():
    print("Loading Owt dataset...")
    # Load the dataset from the specified path
    dataset = load_dataset("Skylion007/openwebtext", split="train")
    return dataset


In [ ]:
load_owt()

In [ ]:
def load_wp():
    print("Loading Writing Prompts dataset...")
    # Load the dataset from the specified path
    dataset = load_dataset("llm-aes/writing-prompts", split="train")
    return dataset


load_wp()

# notoxic

In [ ]:
from dataset import get_notoxic_dataset

dataset =get_notoxic_dataset('xsum-processed')

In [ ]:
from text_filter import StructuralArtifactFilter, TextWordLengthTruncator

word_length_truncator = TextWordLengthTruncator(min_length=150, max_length=300)
artifact_filter_truncator = StructuralArtifactFilter()

word_length_dataset = dataset.filter(word_length_truncator.filter, batched=True, batch_size=64, load_from_cache_file=False)
new_dataset = word_length_dataset.filter(artifact_filter_truncator.filter, batched=True, batch_size=64, load_from_cache_file=False)


In [ ]:
new_dataset = new_dataset.select(range(100))

In [ ]:
from text_filter import NotAcceptableFilter


na_filter = NotAcceptableFilter()
na_dataset = new_dataset.filter(na_filter.filter, batched=True, batch_size=64, load_from_cache_file=False)

In [ ]:
na_dataset

# WP


In [ ]:
import sys
sys.path.append('../')
from utils.dataset import DatasetLoader


dataset_loader = DatasetLoader("~/.datasets")
dataset_loader.load("wp-processed_notoxic")


In [ ]:
from utils.text_filter import TextWordLengthTruncator
word_length_filter = TextWordLengthTruncator(
            min_length=150,
            max_length=500
)

In [ ]:

dataset_loader.dataset.filter(word_length_filter.filter, batched=True, batch_size=64, load_from_cache_file=False)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("euclaise/writingprompts", split="train")

In [ ]:
dataset

In [ ]:
dataset = load_dataset("EdinburghNLP/xsum", split="train")

In [ ]:
dataset['document']

In [ ]:
import sys
sys.path.append('../')

In [ ]:
from hydra import compose, initialize
from omegaconf import OmegaConf

with initialize(version_base=None, config_path="../conf"):
    cfg = compose(config_name="config")

print(cfg.experiment.ranges)

In [ ]:
from datasets import load_dataset, load_from_disk
# dataset = load_from_disk("~/.datasets/xsum-processed_notoxic")
dataset = load_dataset("euclaise/writingprompts", split="train")

In [ ]:
from utils.range_classification import add_range_to_dataset

dataset = dataset.rename_column("story", "text")

range_dataset = add_range_to_dataset(dataset, 
                     cfg.experiment.ranges.short.range, 
                     cfg.experiment.ranges.medium.range, 
                        cfg.experiment.ranges.long.range)

In [ ]:
from utils.sampling import stratified_random_sampling_by_range



sampled_dataset = stratified_random_sampling_by_range(range_dataset,
                                                      cfg.experiment.ranges)

In [ ]:
sampled_dataset

In [ ]:
from utils.dataset import save_texts_line_by_line


save_texts_line_by_line(sampled_dataset, "wp_sampled.txt", text_column="text")

In [ ]:
from datasets import load_dataset, Dataset

dataset_stream = load_dataset(
    "Skylion007/openwebtext",
    split="train",
    streaming=True
)

dataset_500k = Dataset.from_list(
    list(dataset_stream.take(500_000))
)


In [ ]:
from utils.range_classification import add_range_to_dataset


range_dataset = add_range_to_dataset(dataset_500k, 
                     cfg.experiment.ranges.short.range, 
                     cfg.experiment.ranges.medium.range, 
                        cfg.experiment.ranges.long.range)

In [ ]:
sampled_dataset = stratified_random_sampling_by_range(range_dataset,
                                                      cfg.experiment.ranges)

In [ ]:
from utils.dataset import save_texts_line_by_line


save_texts_line_by_line(sampled_dataset, "owt_sampled.txt", text_column="text")

# Aggregate txt with other info and create a new dataset

In [1]:
import sys
sys.path.append('../')
from utils.filehandler import FileHandler


file_handler = FileHandler("outputs/owt_sampled.txt", mode='r')
lines = file_handler.get_lines()
file_handler.close_file()

File already exists: outputs/owt_sampled.txt
Opened file: outputs/owt_sampled.txt in mode: r
Retrieved 600 lines from file: outputs/owt_sampled.txt
Closed file: outputs/owt_sampled.txt


In [2]:
from datasets import load_dataset, Dataset

dataset_stream = load_dataset(
    "Skylion007/openwebtext",
    split="train",
    streaming=True
)

dataset_500k = Dataset.from_list(list(dataset_stream.take(500_000)))


/mnt/hdd/git/LLM-detection/when-human-write-like-machines/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import numpy as np
import torch
import faiss
from sentence_transformers import SentenceTransformer

# -----------------------------
# 1. Modello embedding su GPU
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

def normalize_text(s):
    return " ".join(str(s).replace("\n", " ").replace("\r", " ").split())

lines_clean = [normalize_text(x) for x in lines]
texts_clean = [normalize_text(x) for x in dataset_500k["text"]]

# -----------------------------
# 2. Embeddings delle righe target
# -----------------------------
line_emb = model.encode(
    lines_clean,
    batch_size=256,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

# -----------------------------
# 3. FAISS index
#    Con embeddings normalizzati,
#    inner product = cosine similarity
# -----------------------------
dim = line_emb.shape[1]

cpu_index = faiss.IndexFlatIP(dim)

if device == "cuda":
    res = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
else:
    index = cpu_index

index.add(line_emb)

# -----------------------------
# 4. Embeddings del dataset
# -----------------------------
text_emb = model.encode(
    texts_clean,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

# -----------------------------
# 5. Cerca il match semantico più vicino
# -----------------------------
scores, indices = index.search(text_emb, k=1)

threshold = 0.90
keep_mask = scores[:, 0] >= threshold

# -----------------------------
# 6. Filtra dataset
# -----------------------------
keep_indices = np.where(keep_mask)[0].tolist()

filtered_dataset = dataset_500k.select(keep_indices)

Batches: 100%|██████████| 7813/7813 [05:50<00:00, 22.31it/s]


In [5]:
filtered_dataset
from collections import Counter

matched_line_indices = indices[keep_mask, 0]
counts = Counter(matched_line_indices)

duplicates = {idx: c for idx, c in counts.items() if c > 1}

print("Record filtrati:", len(keep_indices))
print("Line uniche matchate:", len(counts))
print("Line con più match:", len(duplicates))
print(duplicates)

Record filtrati: 602
Line uniche matchate: 600
Line con più match: 2
{np.int64(301): 2, np.int64(371): 2}


In [6]:
for line_idx, count in duplicates.items():
    print("=" * 100)
    print("LINE IDX:", line_idx, "COUNT:", count)
    print("TARGET LINE:")
    print(lines_clean[line_idx][:1000])

    matched_dataset_indices = [
        i for i in keep_indices
        if indices[i, 0] == line_idx
    ]

    for i in matched_dataset_indices:
        print("-" * 80)
        print("DATASET IDX:", i)
        print("SCORE:", scores[i, 0])
        print(texts_clean[i][:1000])

LINE IDX: 301 COUNT: 2
TARGET LINE:
Hillary Clinton, Donald Trump (Photo11: Getty Images) (WXIA) – London's Guardian newspaper reported Monday that California Ku Klux Klan grand dragon Will Quigg was endorsing Hillary Clinton for president. Quigg said he was supporting Clinton because he said she has a “hidden agenda.” “She’s telling everybody what they want to hear so she can get elected, because she’s Bill Clinton’s wife, she’s close to the Bushes. Once she’s in the presidency, she’s going to come out and her true colors are going to show,” Quigg said in the Telegraph. “Border policies are going to be put in place. Our second amendment rights that she’s saying she’s against now, she’s not against. She’s just our choice for the presidency.” Last September, Quigg endorsed Donald Trump, sending a tweet that insisted that he was the “only hope we have of getting white America back,” and signing it the “Church of Invisable Empire.” Facebook Twitter Google+ LinkedIn PHOTOS | Hillary Clinto

In [7]:
best_for_line = {}

for dataset_idx in keep_indices:
    line_idx = int(indices[dataset_idx, 0])
    score = float(scores[dataset_idx, 0])

    if line_idx not in best_for_line or score > best_for_line[line_idx][1]:
        best_for_line[line_idx] = (dataset_idx, score)

unique_keep_indices = [v[0] for v in best_for_line.values()]

filtered_dataset_unique = dataset_500k.select(unique_keep_indices)

print("Prima:", len(filtered_dataset))
print("Dopo dedup 1-per-line:", len(filtered_dataset_unique))

Prima: 602
Dopo dedup 1-per-line: 600


In [9]:
filtered_dataset_unique.save_to_disk("outputs/owt_sampled")

Saving the dataset (1/1 shards): 100%|██████████| 600/600 [00:00<00:00, 172937.22 examples/s]


In [10]:
import numpy as np
import torch
import faiss
from sentence_transformers import SentenceTransformer


def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def normalize_text(s):
    return " ".join(
        str(s)
        .replace("\n", " ")
        .replace("\r", " ")
        .split()
    )


def load_embedding_model(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    device=None
):
    if device is None:
        device = get_device()

    model = SentenceTransformer(model_name, device=device)
    return model, device


def encode_texts(
    texts,
    model,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True
):
    texts_clean = [normalize_text(x) for x in texts]

    embeddings = model.encode(
        texts_clean,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=normalize_embeddings,
        show_progress_bar=show_progress_bar
    ).astype("float32")

    return texts_clean, embeddings


def build_faiss_index(embeddings, device="cpu"):
    dim = embeddings.shape[1]

    cpu_index = faiss.IndexFlatIP(dim)

    if device == "cuda":
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
    else:
        index = cpu_index

    index.add(embeddings)

    return index


def semantic_search(
    query_embeddings,
    index,
    k=1
):
    scores, indices = index.search(query_embeddings, k)
    return scores, indices


def filter_by_semantic_similarity(
    dataset,
    dataset_embeddings,
    index,
    threshold=0.90,
    text_field="text"
):
    scores, indices = semantic_search(
        query_embeddings=dataset_embeddings,
        index=index,
        k=1
    )

    keep_mask = scores[:, 0] >= threshold
    keep_indices = np.where(keep_mask)[0].tolist()

    filtered_dataset = dataset.select(keep_indices)

    return filtered_dataset, keep_indices, scores, indices, keep_mask


def deduplicate_one_match_per_line(
    dataset,
    keep_indices,
    scores,
    indices
):
    best_for_line = {}

    for dataset_idx in keep_indices:
        line_idx = int(indices[dataset_idx, 0])
        score = float(scores[dataset_idx, 0])

        if line_idx not in best_for_line or score > best_for_line[line_idx][1]:
            best_for_line[line_idx] = (dataset_idx, score)

    unique_keep_indices = [v[0] for v in best_for_line.values()]

    filtered_dataset_unique = dataset.select(unique_keep_indices)

    return filtered_dataset_unique, unique_keep_indices, best_for_line


def print_match_examples(
    dataset,
    lines_clean,
    texts_clean,
    unique_keep_indices,
    scores,
    indices,
    n=10
):
    for dataset_idx in unique_keep_indices[:n]:
        line_idx = int(indices[dataset_idx, 0])
        score = float(scores[dataset_idx, 0])

        print("=" * 100)
        print("DATASET IDX:", dataset_idx)
        print("LINE IDX:", line_idx)
        print("SCORE:", score)

        print("\nDATASET TEXT:")
        print(texts_clean[dataset_idx][:1000])

        print("\nMATCHED LINE:")
        print(lines_clean[line_idx][:1000])

In [11]:
model, device = load_embedding_model(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Device:", device)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 23006.35it/s]


Device: cuda


In [12]:
index = build_faiss_index(
    embeddings=line_emb,
    device=device
)

### xsum

In [20]:
dataset = load_dataset("EdinburghNLP/xsum", split="train")
dataset = dataset.rename_column("document", "text")
file_handler = FileHandler("outputs/xsum_sampled.txt", mode='r')
lines = file_handler.get_lines()
file_handler.close_file()

File already exists: outputs/xsum_sampled.txt
Opened file: outputs/xsum_sampled.txt in mode: r
Retrieved 600 lines from file: outputs/xsum_sampled.txt
Closed file: outputs/xsum_sampled.txt


In [19]:
def semantic_filter_dataset_against_lines(
    dataset,
    lines,
    text_field="text",
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    threshold=0.90,
    line_batch_size=256,
    dataset_batch_size=64,
    deduplicate=True
):
    model, device = load_embedding_model(model_name=model_name)

    print("Device:", device)

    lines_clean, line_emb = encode_texts(
        texts=lines,
        model=model,
        batch_size=line_batch_size,
        show_progress_bar=True
    )

    index = build_faiss_index(
        embeddings=line_emb,
        device=device
    )

    texts_clean, text_emb = encode_texts(
        texts=dataset[text_field],
        model=model,
        batch_size=dataset_batch_size,
        show_progress_bar=True
    )

    filtered_dataset, keep_indices, scores, indices, keep_mask = filter_by_semantic_similarity(
        dataset=dataset,
        dataset_embeddings=text_emb,
        index=index,
        threshold=threshold,
        text_field=text_field
    )

    if deduplicate:
        filtered_dataset_unique, unique_keep_indices, best_for_line = deduplicate_one_match_per_line(
            dataset=dataset,
            keep_indices=keep_indices,
            scores=scores,
            indices=indices
        )

        result = filtered_dataset_unique
        final_indices = unique_keep_indices
    else:
        best_for_line = None
        result = filtered_dataset
        final_indices = keep_indices

    metadata = {
        "device": device,
        "threshold": threshold,
        "num_lines": len(lines),
        "num_dataset_rows": len(dataset),
        "num_matches_before_dedup": len(filtered_dataset),
        "num_matches_after_dedup": len(result),
        "keep_indices": keep_indices,
        "final_indices": final_indices,
        "scores": scores,
        "indices": indices,
        "keep_mask": keep_mask,
        "lines_clean": lines_clean,
        "texts_clean": texts_clean,
        "best_for_line": best_for_line
    }

    return result, metadata

In [22]:
result, metadata = semantic_filter_dataset_against_lines(dataset, lines)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16303.62it/s]


Device: cuda


Batches: 100%|██████████| 3189/3189 [01:48<00:00, 29.36it/s]


In [25]:
result.save_to_disk("outputs/xsum_sampled")

Saving the dataset (1/1 shards): 100%|██████████| 600/600 [00:00<00:00, 111136.83 examples/s]


### wp

In [26]:

dataset = load_dataset("euclaise/writingprompts", split="train")
dataset = dataset.rename_column("story", "text")
file_handler = FileHandler("outputs/wp_sampled.txt", mode='r')
lines = file_handler.get_lines()
file_handler.close_file()

File already exists: outputs/wp_sampled.txt
Opened file: outputs/wp_sampled.txt in mode: r
Retrieved 600 lines from file: outputs/wp_sampled.txt
Closed file: outputs/wp_sampled.txt


In [27]:
result, metadata = semantic_filter_dataset_against_lines(dataset, lines)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15261.71it/s]


Device: cuda


Batches: 100%|██████████| 4260/4260 [02:36<00:00, 27.19it/s]


In [29]:
result.save_to_disk("outputs/wp_sampled")

Saving the dataset (1/1 shards): 100%|██████████| 600/600 [00:00<00:00, 95238.51 examples/s] 


In [1]:
import sys
sys.path.append('../')
from datasets import load_from_disk

dataset = load_from_disk("output/xsum_sampled")

/mnt/hdd/git/LLM-detection/when-human-write-like-machines/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
dataset['text'][16]
# dataset['summary'][17]

"Earlier, Cook had made a battling 52 for the visitors but Essex lost 8-69 as they were bowled out for 129, with Roelof van der Merwe taking 3-26.\nNeil Wagner then took 6-48 as the hosts faltered, but Jack Leach and Craig Overton's 50-run last-wicket stand propelled Somerset to 174 all out.\nCook and Nick Browne survived a testing seven overs as Essex closed on 10-0.\nLeft-hander Cook resumed his first innings on 39 and looked uneasy at the crease as he slashed at a ball from spinner Leach, sending it just wide of Lewis Gregory at first slip.\nA misfield allowed the 32-year-old to scamper through for his half-century, which came off 85 balls with nine fours, but was bowled via an inside edge shortly afterwards by Gregory.\nSomerset's young English quartet of Craig and Jamie Overton, Leach and Gregory all impressed with the ball as the away side were bundled out still trailing by 80 runs.\nFollowing lunch, Wagner collected his maiden five-wicket haul for Essex as he attacked the Somers